# Step 3: Generation (LLM + Prompt Engineering)

**Goal:** Wire up the LLM, craft a system prompt that handles edge cases, and test the full pipeline.

**What we know from Step 2:**
- Retrieval works. 6/7 positive questions find the right source file.
- Threshold at 0.55 is correct.
- Two edge cases need the LLM to handle them:
  - **Hybrid work:** Context is retrieved (remote work pages). LLM must explain GitLab is all-remote, not hybrid.
  - **Stock price:** Context is retrieved (public company page). LLM must recognise the context doesn't contain the stock price.

**What we're building:**
1. System prompt that prevents hallucination
2. Prompt construction: context + chat history + user question
3. Full pipeline test against golden dataset
4. Edge case stress testing

In [1]:
!pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)


In [4]:
!pip install litellm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 50.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.0/994.0 kB 37.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [litellm]m4/5 [litellm]


In [2]:
import os
from dotenv import load_dotenv

In [3]:
# !pip install litellm chromadb
from dotenv import load_dotenv
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [5]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
import litellm

# Set your API key (or export GEMINI_API_KEY in terminal before launching jupyter)
# os.environ["GEMINI_API_KEY"] = "your-key-here"

# ── Load ChromaDB ──
CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "gitlab_handbook"
ef = embedding_functions.DefaultEmbeddingFunction()
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"}
)
print(f"Collection loaded: {collection.count()} chunks")

Collection loaded: 802 chunks


In [6]:
# ── Retrieval function (carried over from Step 2) ──
TOP_K = 5
RELEVANCE_THRESHOLD = 0.55

def retrieve(query: str, top_k: int = TOP_K, threshold: float = RELEVANCE_THRESHOLD):
    results = collection.query(query_texts=[query], n_results=top_k)
    context_parts = []
    sources = []

    for i in range(len(results["documents"][0])):
        doc = results["documents"][0][i]
        source = results["metadatas"][0][i]["source"]
        distance = results["distances"][0][i]

        if distance <= threshold:
            context_parts.append(f"[Source: {source}]\n{doc}")
            if source not in sources:
                sources.append(source)

    context_text = "\n\n---\n\n".join(context_parts) if context_parts else ""
    return context_text, sources

## 3.1 The System Prompt

This is the most important piece of text in the entire project. It controls how the LLM uses (or refuses to use) the retrieved context.

**Key design decisions:**

1. **"Answer ONLY based on the provided context"** prevents the model from using its general knowledge. Without this, it would happily answer "What is the capital of France?" from its training data.

2. **"If the context does not contain the specific information needed to answer the question, say so clearly"** handles the stock price case. The context will contain info about GitLab being a public company, but not the actual stock price. The LLM needs to distinguish between "I have related context" and "I have the answer."

3. **"Do not speculate or infer beyond what is explicitly stated"** prevents the model from connecting dots that aren't there. If the handbook says "GitLab is all-remote," the model shouldn't infer a hybrid work policy from that.

4. **"Mention which handbook page(s) your answer comes from"** is free since we already pass source filenames in the context. Makes the answer more trustworthy and satisfies the citation requirement.

In [7]:
MODEL = "gemini/gemma-3-27b-it"

SYSTEM_PROMPT = """You are a helpful assistant that answers questions about the GitLab Handbook.

RULES:
1. Answer ONLY based on the provided context from the GitLab Handbook. Do not use any outside knowledge.
2. If the context does not contain the specific information needed to answer the question, say so clearly. For example, if someone asks about a stock price but the context only discusses GitLab's history as a public company, explain that the handbook covers that topic generally but does not contain the specific information requested.
3. Do not speculate or infer beyond what is explicitly stated in the context.
4. Be concise and direct.
5. Mention which handbook page(s) your answer comes from, using the source filenames provided in the context.
6. If no context is provided at all, tell the user that you couldn't find relevant information in the GitLab Handbook for their question."""

## 3.2 Prompt Construction

The user prompt combines three things:
1. **Retrieved context** from ChromaDB (or a note that none was found)
2. **Chat history** for conversation continuity (last 3 exchanges)
3. **The user's question**

We keep these clearly labelled so the LLM can distinguish between them.

In [8]:
def format_chat_history(history: list) -> str:
    """Format chat history into a readable string.
    Handles both tuple format [(user, bot), ...] and dict format [{role, content}, ...].
    """
    if not history:
        return "(No previous conversation)"
    lines = []
    for item in history[-6:]:  # last 3 exchanges max
        if isinstance(item, (list, tuple)) and len(item) == 2:
            lines.append(f"User: {item[0]}")
            if item[1]:
                lines.append(f"Assistant: {item[1]}")
        elif isinstance(item, dict):
            role = "User" if item["role"] == "user" else "Assistant"
            lines.append(f"{role}: {item['content']}")
    return "\n".join(lines) if lines else "(No previous conversation)"


def build_prompt(question: str, context: str, chat_history: list = None) -> str:
    """Construct the full user prompt sent to the LLM."""
    history_str = format_chat_history(chat_history or [])

    if context:
        return f"""CONTEXT FROM GITLAB HANDBOOK:
{context}

CONVERSATION HISTORY:
{history_str}

USER QUESTION: {question}

Answer the question using ONLY the context above. If the context doesn't contain the specific answer, say so."""
    else:
        return f"""No relevant context was found in the GitLab Handbook for this question.

CONVERSATION HISTORY:
{history_str}

USER QUESTION: {question}

Tell the user that you couldn't find relevant information in the GitLab Handbook to answer their question."""

## 3.3 Generation Function

Puts it all together: retrieve → build prompt → call LLM → append citations.

In [9]:
def ask(question: str, chat_history: list = None, verbose: bool = False) -> str:
    """
    Full RAG pipeline: retrieve context, build prompt, call LLM, return answer with citations.
    
    Set verbose=True to see retrieval details and the full prompt (useful for debugging).
    """
    # 1. Retrieve
    context, sources = retrieve(question)
    
    if verbose:
        print(f"[Retrieval] Sources: {sources}")
        print(f"[Retrieval] Context length: {len(context)} chars")
    
    # 2. Build prompt
    prompt = build_prompt(question, context, chat_history)
    
    if verbose:
        print(f"[Prompt] Total length: {len(prompt)} chars")
        print("-" * 40)
    
    # 3. Call LLM
    try:
        response = litellm.completion(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            temperature=0.3,
            max_tokens=1024,
        )
        answer = response.choices[0].message.content.strip()
    except Exception as e:
        answer = f"Error calling LLM: {str(e)}"
    
    # 4. Append source citations
    if sources:
        citation_block = "\n\n---\n**Sources:**\n" + "\n".join(f"- `{s}`" for s in sources)
        answer += citation_block
    
    return answer

## 3.4 Smoke Test

Before running the full evaluation, let's test a few queries manually to make sure the pipeline works end-to-end.

In [10]:
# Simple positive case
print(ask("What does handbook first mean?", verbose=True))

[Retrieval] Sources: ['about_handbook-usage.md']
[Retrieval] Context length: 4832 chars
[Prompt] Total length: 5070 chars
----------------------------------------
According to the GitLab Handbook, “handbook first” means that you should communicate a proposed change through a change to the handbook, rather than using presentations, emails, or chat messages. It ensures there is no duplication, the handbook is always up to date, and others are better able to contribute. It also means defaulting to the public handbook for information that can be made public. (about_handbook-usage.md)

---
**Sources:**
- `about_handbook-usage.md`


In [11]:
# The hybrid work edge case — should use remote work context, not hallucinate
print(ask("What is GitLab's hybrid work policy?", verbose=True))

[Retrieval] Sources: ['company_culture_all-remote_getting-started.md', 'company_culture_all-remote_asynchronous.md', 'business-technology_entapps-documentation_policies_gitlab-business-continuity-plan.md']
[Retrieval] Context length: 4853 chars
[Prompt] Total length: 5097 chars
----------------------------------------
The provided context does not contain information about GitLab's hybrid work policy. It does state that GitLab has a “100% remote culture” and that those who thrive there “embrace a liberating and empowering set of values” and operate differently than in conventional corporations (company_culture_all-remote_getting-started.md). However, it does not discuss a hybrid work model.

---
**Sources:**
- `company_culture_all-remote_getting-started.md`
- `company_culture_all-remote_asynchronous.md`
- `business-technology_entapps-documentation_policies_gitlab-business-continuity-plan.md`


In [12]:
# Negative case — no context at all
print(ask("What is the capital of France?", verbose=True))

[Retrieval] Sources: []
[Retrieval] Context length: 0 chars
[Prompt] Total length: 276 chars
----------------------------------------
I couldn't find relevant information in the GitLab Handbook for your question.


In [13]:
# Negative case — related context exists but doesn't answer the question
print(ask("What is GitLab's current stock price?", verbose=True))

[Retrieval] Sources: ['company_being-a-public-company.md', 'communication_confidentiality-levels.md']
[Retrieval] Context length: 6229 chars
[Prompt] Total length: 6474 chars
----------------------------------------
The handbook covers GitLab's history as a public company and discusses the importance of focusing on KPIs to drive stock price in the long run, but it does not contain the current stock price. [company_being-a-public-company.md]

---
**Sources:**
- `company_being-a-public-company.md`
- `communication_confidentiality-levels.md`


## 3.5 Chat History Test

The assignment requires continued conversations. Let's verify follow-up questions work.

In [14]:
# Simulate a conversation
history = []

# Turn 1
q1 = "What does handbook first mean?"
a1 = ask(q1, chat_history=history)
print(f"User: {q1}")
print(f"Bot: {a1}")
print("\n" + "=" * 60 + "\n")

# Add to history
history.append((q1, a1))

# Turn 2 — follow-up that relies on context from Turn 1
q2 = "Why is that approach better than using Slack or email?"
a2 = ask(q2, chat_history=history)
print(f"User: {q2}")
print(f"Bot: {a2}")

User: What does handbook first mean?
Bot: According to the GitLab Handbook, “handbook first” means there is no duplication of information, the handbook is always up to date, and others are better able to contribute. It ensures that changes are communicated through the handbook itself—not presentations, emails, or chat messages—to maintain context and transparency. (about_handbook-usage.md)

It also means defaulting to the public handbook for information that can be made public, ensuring everyone has access to safely shared information. (about_handbook-usage.md)

---
**Sources:**
- `about_handbook-usage.md`


User: Why is that approach better than using Slack or email?
Bot: The handbook states that chat-style communication tools like Slack should be used primarily for informal communication. If you are used to Slack being an always-on center of urgency, breaking that reliance will require deliberate effort. Clearing all messages daily or weekly can also reduce mental load. (company_cult

## 3.6 Full Golden Dataset Evaluation

Run every golden dataset question through the full pipeline and save results.

This produces `test_results.json`, one of your deliverables.

In [15]:
import time

with open("golden_dataset.json", "r") as f:
    golden = json.load(f)

results = []

for item in golden:
    question = item["question"]
    print(f"[Q{item['id']}] {question}")
    
    context, sources = retrieve(question)
    answer = ask(question)
    
    results.append({
        "id": item["id"],
        "question": question,
        "type": item["type"],
        "expected_answer": item["expected_answer"],
        "chatbot_answer": answer,
        "retrieved_sources": sources,
        "context_found": bool(context)
    })
    
    print(f"  Sources: {sources}")
    print(f"  Answer preview: {answer[:150]}...")
    print()
    
    time.sleep(1)  # rate limit safety

# Save
with open("test_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("=" * 60)
print(f"Saved {len(results)} results to test_results.json")

[Q1] What does 'handbook first' mean at GitLab?
  Sources: ['about_handbook-usage.md', 'about_direction.md', 'communication_confidentiality-levels.md']
  Answer preview: Having a **"handbook first"** mentality ensures there is no duplication; the handbook is always up to date, and others are better able to contribute. ...

[Q2] What is the process for proposing changes to the GitLab handbook?
  Sources: ['about_handbook-usage.md', 'about_direction.md']
  Answer preview: Here's the process for proposing changes to the GitLab handbook, according to the provided context:

1. A proposal is made in a merge request to the h...

[Q3] What is the GitLab Handbook and who is it for?
  Sources: ['about_direction.md', 'about_handbook-usage.md']
  Answer preview: The GitLab Handbook is the single source of truth for how GitLab operates, including processes, policies, and product direction ([about_direction.md](...

[Q4] When should an issue be escalated via the handbook escalation process?
  Source

## 3.7 Review Results

Print each result in full so you can verify the quality of the answers.

In [16]:
with open("test_results.json", "r") as f:
    results = json.load(f)

for r in results:
    print(f"Q{r['id']} [{r['type'].upper()}]: {r['question']}")
    print(f"Sources: {r['retrieved_sources']}")
    print(f"\nExpected:\n{r['expected_answer']}")
    print(f"\nGot:\n{r['chatbot_answer']}")
    print("\n" + "=" * 60 + "\n")

Q1 [POSITIVE]: What does 'handbook first' mean at GitLab?
Sources: ['about_handbook-usage.md', 'about_direction.md', 'communication_confidentiality-levels.md']

Expected:
Handbook first means documenting in the handbook before taking any action. Instead of communicating changes through presentations, emails, or chat messages, team members should propose changes via merge requests to the handbook. This ensures there is no duplication, the handbook is always up to date, and others can better contribute.

Got:
Having a **"handbook first"** mentality ensures there is no duplication; the handbook is always up to date, and others are better able to contribute. (about_handbook-usage.md)

---
**Sources:**
- `about_handbook-usage.md`
- `about_direction.md`
- `communication_confidentiality-levels.md`


Q2 [POSITIVE]: What is the process for proposing changes to the GitLab handbook?
Sources: ['about_handbook-usage.md', 'about_direction.md']

Expected:
Process changes should be communicated by lin

## What to Check

**Positive questions:** Does the answer match the expected answer in substance? It doesn't need to be word-for-word, just accurate and grounded in the right source.

**Hybrid work:** Does the LLM say something like "GitLab is an all-remote company" rather than making up a hybrid policy? This is the test of the system prompt.

**Capital of France:** Should refuse cleanly. No context was retrieved, so the LLM should say it has no relevant information.

**Stock price:** Should acknowledge the public company page exists but say the handbook doesn't contain stock price information. This is the hardest test.

**If the stock price answer hallucinates:** Tighten the system prompt. Add a more explicit instruction like: "If the retrieved context is about a related topic but does not contain the specific fact the user is asking for, explain what the handbook does cover and what it does not."

---

**Next:** Step 4 wraps this in a Gradio UI and deploys to Hugging Face Spaces.